<div align="center">
  <img src="https://raw.githubusercontent.com/NaumanHSA/neurosurfer/main/docs/assets/banner/neurosurfer_banner_white.png" alt="Neurosurfer" width="45%"/>
</div>

<br/>

# 01 — Providers, Agents & RAG

This notebook introduces the three building blocks of Neurosurfer:

1. **LLM Provider** — talk directly to the model (`complete` / `stream`)
2. **Agents** — three types that build on the provider:
   - `Agent` — one-shot call, optional structured output
   - `AgenticLoop` — multi-step autonomous loop with native tool calling
   - `ReactAgent` — text-parsing ReAct loop for models without native tool calling
3. **RAG** — ingest a PDF, retrieve relevant chunks, generate a grounded answer

**Model used:** `google/gemma-4-12b-qat` served via LM Studio on `localhost:1234`

---

## Contents
1. [Setup](#1-setup)
2. [LLM Provider — direct usage](#2-llm-provider--direct-usage)
3. [Agent — one-shot](#3-agent--one-shot)
4. [AgenticLoop — multi-step tool use](#4-agenticloop--multi-step-tool-use)
5. [ReactAgent — text-parsing ReAct](#5-reactagent--text-parsing-react)
6. [RAG — Retrieval-Augmented Generation](#6-rag--retrieval-augmented-generation)

---

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Point Python at the repo root when running from tutorials/
repo_root = Path(os.getcwd()).parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import neurosurfer
print(f"neurosurfer {neurosurfer.__version__}")

In [ ]:
# ── LM Studio connection ──────────────────────────────────────────────────────
# Make sure LM Studio is running with the model loaded and the local server ON.
# Server → default port 1234.  Copy the model identifier from the LM Studio UI.

LM_STUDIO_URL   = "http://localhost:1234/v1"
LM_STUDIO_MODEL = "qwen/qwen3.5-9b"   # adjust if your name differs
CONTEXT_WINDOW  = 48_768                       # check your model's context length

# Path to the sample document used in the RAG section
PDF_PATH = repo_root / "data" / "pixelrag-paper.pdf"
print(f"PDF exists: {PDF_PATH.exists()} → {PDF_PATH}")

---

## 2. LLM Provider — direct usage

The **provider** is the lowest layer — a thin async adapter over the model's API.  
You can use it directly when you don't need agent logic (history management, tool gating, events).

Neurosurfer ships two built-in providers:

| Class | Use with |
|---|---|
| `AnthropicProvider` | Anthropic Claude (cloud) |
| `OpenAICompatProvider` | OpenAI, LM Studio, Ollama, vLLM, llama.cpp — anything on the `/v1` API |

In [ ]:
from neurosurfer.llm.providers.openai import OpenAICompatProvider

provider = OpenAICompatProvider(
    model          = LM_STUDIO_MODEL,
    base_url       = LM_STUDIO_URL,
    api_key        = "lm-studio",   # LM Studio ignores this; any non-empty string works
    context_window = CONTEXT_WINDOW,
)

print(f"Provider: {provider.model}")
print(f"Context window: {provider.capabilities.context_window:,} tokens")
print(f"Native tool calling: {provider.capabilities.tool_call_style}")

### 2a. Single completion

`provider.complete(messages, system, tools, config)` sends one request and returns a `CanonicalResponse`.

- `messages` — list of `Message` objects (conversation history)
- `system` — the system prompt string (passed out-of-band)
- `tools` — tool schemas (empty list = no tools)
- `config` — `GenerationConfig` (max_tokens, temperature, …)

In [ ]:
from neurosurfer.llm.types import Message, GenerationConfig

messages = [
    Message.user_text("What is Retrieval-Augmented Generation? Answer in 2 sentences.")
]
config = GenerationConfig(max_tokens=1024, temperature=0.7)

response = await provider.complete(
    messages = messages,
    system   = "You are a concise AI assistant.",
    tools    = [],
    config   = config,
)

print("Answer:", response.text())
print(f"\nTokens used — input: {response.usage.input_tokens}, output: {response.usage.output_tokens}")

### 2b. Streaming

`provider.stream()` is an async generator that yields typed events token by token.  
This is how you get live output — same as streaming from the chat UI.

In [ ]:
from neurosurfer.llm.types import TextDelta, Done
from IPython.display import display, Markdown, clear_output

messages = [Message.user_text("Explain vector embeddings in 3 bullet points.")]
config   = GenerationConfig(max_tokens=1024, temperature=0.7)

output = ""
md = display(Markdown(""), display_id=True)

async for event in provider.stream(
    messages = messages,
    system   = "You are a concise AI assistant.",
    tools    = [],
    config   = config,
):
    if isinstance(event, TextDelta):
        output += event.text
        md.update(Markdown(output))
    elif isinstance(event, Done):
        print(f"\n\nDone. Tokens: {event.response.usage.input_tokens} in / {event.response.usage.output_tokens} out")

---

## 3. Agent — one-shot

`Agent` makes **one bounded call** — optionally with tools (one round) or a structured Pydantic output.  
It's the simplest agent: ask once, get an answer back.

### Constructing agents

All three agent types share the same base constructor:

| Parameter | Type | Purpose |
|---|---|---|
| `provider` | `Provider` | The LLM backend |
| `tools` | `ToolPool` | Available tools (use `default_pool()` for all built-ins) |
| `system_prompt` | `str` | The agent's persona / instructions |
| `guardrails` | `Guardrails` | Safety limits (write scope, shell policy, max turns) |
| `io` | `IOHandler` | How to ask the user for approvals |
| `cwd` | `Path` | Working directory for file tools |

In [ ]:
from neurosurfer.agents import Agent, Guardrails
from neurosurfer.tools import default_pool, AutoApproveIOHandler

agent = Agent(
    provider      = provider,
    tools         = default_pool(),
    system_prompt = "You are a helpful assistant. Be concise.",
    guardrails    = Guardrails(),   # defaults: write_scope=["**"], max_turns=200
    io            = AutoApproveIOHandler(),
    verbose       = False,   # this cell renders events manually
    cwd           = repo_root,
)

print("Agent ready:", type(agent).__name__)

### 3a. Text response

`agent.complete(user_input)` runs to completion and returns the final text directly.

In [ ]:
from IPython.display import display, Markdown, clear_output

answer = await agent.complete("What is the capital of France, and what is it famous for?")
display(Markdown(answer))

### 3b. Structured output

Pass `output_schema=YourModel` at construction time and the agent returns a validated Pydantic model — no JSON parsing, no repair loops needed.  
Neurosurfer uses **native tool-use under the hood** to guarantee valid JSON output.

In [ ]:
from pydantic import BaseModel

class MovieReview(BaseModel):
    title: str
    genre: str
    rating: float          # 0.0 – 10.0
    summary: str
    pros: list[str]
    cons: list[str]

structured_agent = Agent(
    provider      = provider,
    tools         = default_pool(),
    system_prompt = "You are a film critic. Return structured reviews.",
    guardrails    = Guardrails(),
    io            = AutoApproveIOHandler(),
    verbose       = False,   # this cell renders events manually
    cwd           = repo_root,
    output_schema = MovieReview,   # ← structured output
)

review: MovieReview = await structured_agent.complete("Write a review for the movie Inception (2010).")

print(f"Title:   {review.title}")
print(f"Genre:   {review.genre}")
print(f"Rating:  {review.rating}/10")
print(f"Summary: {review.summary}")
print("Pros:")
for p in review.pros:
    print(f"  + {p}")
print("Cons:")
for c in review.cons:
    print(f"  - {c}")

---

## 4. AgenticLoop — multi-step tool use

`AgenticLoop` is the **production multi-step agent**. It runs a loop:

```
while not finished:
    call model → if tool_use: execute tools → append results → repeat
```

It uses the provider's **native function-calling API** — no text parsing, no sentinel leakage.  
The loop ends when the model calls the `finish` tool or reaches `max_turns`.

### Events

The `run()` method is an async generator that emits typed events:

| Event | When |
|---|---|
| `TextDelta` | A chunk of text from the model |
| `ThinkingDelta` | A thinking/reasoning chunk (provider-dependent) |
| `TurnCompleted` | One model call finished (includes token usage) |
| `ToolStarted` | A tool is about to execute |
| `ToolFinished` | A tool returned its result |
| `RunFinished` | The entire run is done (status: completed / max_turns / …) |

In [ ]:
from neurosurfer.agents import AgenticLoop, events

loop_agent = AgenticLoop(
    provider      = provider,
    tools         = default_pool(),
    system_prompt = (
        "You are a helpful assistant with access to file and search tools. "
        "Complete the task step by step, then call the finish tool when done."
    ),
    guardrails    = Guardrails(),
    io            = AutoApproveIOHandler(),
    verbose       = False,   # this cell renders events manually
    cwd           = repo_root,
)

print("AgenticLoop ready.")

In [ ]:
# Run a simple task: list files in the data/ directory and describe what's there

task = "List the files inside the 'data' directory, take the first file, read it and summarize its contents briefly."

print(f"Task: {task}\n{'─' * 60}")

async for ev in loop_agent.run(task):

    if isinstance(ev, events.TextDelta):
        print(ev.text, end="", flush=True)

    elif isinstance(ev, events.ToolStarted):
        print(f"\n\n  [→ tool] {ev.name}({ev.args})")

    elif isinstance(ev, events.ToolFinished):
        result_preview = ev.result.content[:120].replace("\n", " ")
        suffix = "..." if len(ev.result.content) > 120 else ""
        print(f"  [← result] {result_preview}{suffix}")

    elif isinstance(ev, events.TurnCompleted):
        print(f"\n  [turn] {ev.usage.input_tokens} in / {ev.usage.output_tokens} out tokens")

    elif isinstance(ev, events.RunFinished):
        print(f"\n\n{'─' * 60}")
        print(f"Run finished — status: {ev.status}")
        if ev.report:
            print(f"Report: {ev.report}")

### 4a. Limiting the tool set

You can give the agent only specific tools instead of the full pool.  
Use `pool.select(["tool_name", ...])` to pick a subset.

In [ ]:
from neurosurfer.tools import default_pool

# Give the agent only read + finish tools — no writes, no shell
read_only_pool = default_pool().select(["read_file", "list_dir", "search", "finish"])
print("Read-only tool pool:", [t.name for t in read_only_pool.all()])

reader_agent = AgenticLoop(
    provider      = provider,
    tools         = read_only_pool,
    system_prompt = "You are a read-only assistant. You can read files but cannot modify them.",
    guardrails    = Guardrails(),
    io            = AutoApproveIOHandler(),
    verbose       = False,   # this cell renders events manually
    cwd           = repo_root,
)

task = "Read the README.md file and give me the top 3 key features of Neurosurfer."
print(f"\nTask: {task}\n{'─' * 60}")

async for ev in reader_agent.run(task):
    if isinstance(ev, events.TextDelta):
        print(ev.text, end="", flush=True)
    elif isinstance(ev, events.ToolStarted):
        print(f"\n  [→ {ev.name}]")
    elif isinstance(ev, events.RunFinished):
        print(f"\n\nDone — {ev.status}")

---

## 5. ReactAgent — text-parsing ReAct

`ReactAgent` implements the **Thought → Action → Observation** loop by *parsing the model's text output* — no native tool-calling API required.

**When to use it:**
- The model doesn't support (or poorly supports) JSON function calling
- You're running a small local model that wasn't trained for tool use
- You want explicit step-by-step reasoning visible in the output

The agent prompts the model to emit:
```
Thought: I need to read the README to understand the project.
Action: read_file
Action Input: {"path": "README.md"}
Observation: <tool result appended here by the agent>
... repeat ...
Thought: I now have enough information to answer.
Final Answer: <the answer>
```

The agent streams the model's reasoning as `ThinkingDelta` events (collapse these to a
"Thinking…" indicator), surfaces each tool call as a `ToolStarted` event with a friendly
`title` (e.g. *"Reading file README.md…"*), and emits only the parsed **Final Answer** as
`TextDelta` — so your UI shows clean progress logs followed by the real answer.

> **Note:** Gemma 4 supports native tool calling, so `AgenticLoop` is the better choice for it.  
> `ReactAgent` is demonstrated here to show the interface — use it when you're running a model that doesn't support function calling.

In [ ]:
from neurosurfer.agents import ReactAgent, Guardrails, events
from neurosurfer.tools import default_pool

react_agent = ReactAgent(
    provider      = provider,
    tools         = default_pool().select(["read_file", "list_dir", "finish"]),
    system_prompt = "You are a helpful assistant. Think step by step before acting.",
    guardrails    = Guardrails(max_turns=10),  # keep it short for the demo
    approval      = "ask",
    verbose       = False,   # this cell renders events manually
    cwd           = repo_root,
)

print("ReactAgent ready.")

In [ ]:
task = "List the files in the current folder, read the important ones, and tell me what this project is about."

print(f"Task: {task}\n{'─' * 60}")

thinking = False  # whether we're currently showing the "Thinking…" indicator

async for ev in react_agent.run(task):

    if isinstance(ev, events.ToolStarted):
        # A tool is about to run — show its friendly, human-readable status line
        # (e.g. "Exploring directory .…", "Reading file README.md…").
        if thinking:
            print()            # close the open "Thinking…" line
            thinking = False
        print(f"🔧 {ev.title or ev.name}")

    elif isinstance(ev, events.ToolFinished):
        preview = ev.result.content[:90].replace("\n", " ")
        suffix = "…" if len(ev.result.content) > 90 else ""
        print(f"   ↳ {preview}{suffix}")

    elif isinstance(ev, events.ThinkingDelta):
        # Intermediate reasoning (Thought/Action scaffolding) — collapse it to a
        # single "Thinking…" indicator instead of dumping the raw text.
        if not thinking:
            print("💭 Thinking…", end="", flush=True)
            thinking = True

    elif isinstance(ev, events.TextDelta):
        # The parsed Final Answer — this is the real, user-facing response. Stream it.
        if thinking:
            print("\n")
            thinking = False
        print(ev.text, end="", flush=True)

    elif isinstance(ev, events.RunFinished):
        print(f"\n\n{'─' * 60}")
        print(f"Run finished — {ev.status}")

---

## 6. RAG — Retrieval-Augmented Generation

RAG lets the model answer questions **grounded in your documents** rather than relying on training data alone.

The pipeline:
```
PDF / file  →  chunk  →  embed  →  vector store
                                        ↓
user query  →  embed  →  similarity search  →  top-k chunks
                                                    ↓
                          LLM(system + chunks + query)  →  answer
```

**What you need:**
```bash
pip install "neurosurfer[rag]"
```
This installs `chromadb`, `sentence-transformers`, and PDF/DOCX/PPTX readers.

### 6a. Initialize the RAG agent

In [ ]:
# The RAG section requires the [rag] extra.
# Run this cell once if you haven't installed it yet, then restart the kernel.
# !pip install "neurosurfer[rag]"

import importlib
if importlib.util.find_spec("chromadb") is None:
    raise ImportError(
        "chromadb not found. Install the rag extra first:\n"
        "  pip install 'neurosurfer[rag]'\n"
        "then restart the kernel and re-run from here."
    )
if importlib.util.find_spec("sentence_transformers") is None:
    raise ImportError(
        "sentence-transformers not found. Install the rag extra first:\n"
        "  pip install 'neurosurfer[rag]'\n"
        "then restart the kernel and re-run from here."
    )
print("RAG dependencies are available — proceeding.")

In [ ]:
from neurosurfer.rag.agent import RAGAgent
from neurosurfer.rag.config import RAGAgentConfig
from neurosurfer.embeddings import _LocalEmbedder
from neurosurfer.vectorstores.chroma import ChromaVectorStore

# ── Embedder ─────────────────────────────────────────────────────────────────
# Converts text → dense vectors.  all-MiniLM-L6-v2 is fast and downloads once.
embedder = _LocalEmbedder(model="all-MiniLM-L6-v2")
print("Embedder loaded.")

# ── Vector store ──────────────────────────────────────────────────────────────
# ChromaDB persists embeddings to disk.  clear_collection=True resets on each run.
vectorstore = ChromaVectorStore(
    collection_name  = "pixelrag-paper",
    clear_collection = True,
    persist_directory= str(repo_root / "tutorials" / "rag-storage"),
)
print("Vector store initialized.")

# ── RAG agent ─────────────────────────────────────────────────────────────────
rag = RAGAgent(
    llm         = provider,
    embedder    = embedder,
    vectorstore = vectorstore,
    config      = RAGAgentConfig(
        top_k              = 6,
        temperature        = 0.4,
        max_new_tokens     = 8192,
        clear_collection_on_init = False,  # we already set clear_collection above
    ),
)
print("RAGAgent ready.")

### 6b. Ingest the document

`rag.ingest(source)` reads the file, splits it into overlapping chunks, embeds each chunk, and stores it in the vector store.  
Only needs to run once per document — the vector store persists to disk.

In [ ]:
result = rag.ingest(PDF_PATH)

print(f"Ingestion complete:")
print(f"  Sources processed : {result.get('sources', 'n/a')}")
print(f"  Chunks created    : {result.get('chunks', 'n/a')}")
print(f"  Chunks stored     : {result.get('added', 'n/a')}")

### 6c. Retrieve — search without generating

`rag.retrieve(query)` returns the most relevant chunks **without** calling the LLM.  
Useful for inspecting what context the model will see, or for using the chunks in your own pipeline.

In [ ]:
query = "How does PixelRAG handle image indexing and retrieval?"

retrieved = rag.retrieve(user_query=query, top_k=4)

print(f"Query: {query}")
print(f"Context tokens used: {retrieved.context_tokens_used}")
print(f"Chunks retrieved: {len(retrieved.docs)}")
print(f"\n{'─' * 60}")
print("Retrieved context (first 800 chars):")
print(retrieved.context[:800], "...")

### 6d. Run — full RAG pipeline

`rag.run(user_prompt)` retrieves the relevant chunks and calls the LLM with them injected into the prompt.  
The model's answer is grounded in the document — it cannot hallucinate facts that aren't there.

In [ ]:
question = "What indexing method does PixelRAG use and what are its advantages over text-only RAG?"

print(f"Question: {question}\n{'─' * 60}")

rag_response = rag.run(user_prompt=question)

print(rag_response.agent_response.text())
print(f"\n{'─' * 60}")
print(f"Tokens: {rag_response.agent_response.usage.input_tokens} in / {rag_response.agent_response.usage.output_tokens} out")

In [ ]:
# Try a few more questions to explore the paper

questions = [
    "What datasets were used to evaluate PixelRAG and what were the key results?",
    "How does PixelRAG compare to baseline methods in terms of retrieval accuracy?",
]

for q in questions:
    print(f"\nQ: {q}")
    print("─" * 60)
    resp = rag.run(user_prompt=q)
    print(resp.agent_response.text())
    print()

---

## Summary

| Class | Use when | Key method |
|---|---|---|
| `OpenAICompatProvider` | You need raw LLM access | `await provider.complete(...)` |
| `Agent` | One call, optional structured output | `await agent.complete(...)` |
| `AgenticLoop` | Multi-step autonomy, native tool calling | `async for ev in agent.run(...)` |
| `ReactAgent` | Text-parsing ReAct, no native function calling | `async for ev in agent.run(...)` |
| `RAGAgent` | Ground answers in your documents | `rag.ingest(...)` then `rag.run(...)` |

---

## What's Next

| # | Notebook | Topic |
|---|---|---|
| **01** | *Providers, Agents & RAG* ← **you are here** | Core building blocks |
| **02** | Custom Tools | Define and register your own tools |
| **03** | Graph Agents | Multi-agent DAG workflows with agents and tools |
| **04** | The Gateway Server | Serve agents as OpenAI-compatible endpoints |